# LOAD DATA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Liberation Serif"], # Matches Times New Roman
    "font.size": 10,                    # Standard for most papers
    "axes.labelsize": 10,
    "xtick.labelsize": 8,              
    "ytick.labelsize": 8,
    "legend.fontsize": 9,
    "text.usetex": False,               
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})
df_deep_train=pd.read_csv("/kaggle/input/multiclass-deep-features/train_features_multiclass.csv")
df_deep_test=pd.read_csv("/kaggle/input/multiclass-deep-features/test_features_multiclass.csv")
df_handcrafted_train=pd.read_csv("/kaggle/input/multiclass-handcrafted-features/Hnadcrafted_Train_features_multiclass.csv")
df_handcrafted_test=pd.read_csv("/kaggle/input/multiclass-handcrafted-features/Handcraftred_Test_features_multiclass.csv")

print("--- Deep Features Labels ---")
print(df_deep_train['label'].value_counts().sort_index())

print("\n--- Handcrafted Features Labels ---")
print(df_handcrafted_train['label'].value_counts().sort_index())

label_fix_map = {0: 3, 1: 2, 2: 1, 3: 0} 

df_deep_train['label'] = df_deep_train['label'].map(label_fix_map)
df_deep_test['label']  = df_deep_test['label'].map(label_fix_map)

print("✅ Labels remapped. Re-checking consistency...")

X_deep= pd.concat((df_deep_train,df_deep_test),axis=0)
X_deep_final = X_deep.drop(columns=['image_id','label'])
y_deep = X_deep['label'].to_numpy().ravel()

X_handcrafted= pd.concat((df_handcrafted_train,df_handcrafted_test),axis=0)
X_handcrafted_final = X_handcrafted.drop(columns=['id','label'])
y_handcrafted = X_handcrafted['label'].to_numpy().ravel()

df_deep_test['id']=df_deep_test['image_id']
df_deep_train['id']=df_deep_train['image_id']
   
df_fusion_train=pd.merge(df_deep_train,df_handcrafted_train,on='id',how='inner')
df_fusion_test=pd.merge(df_deep_test,df_handcrafted_test,on='id',how='inner')

X_fusion_train=df_fusion_train.drop(columns=['image_id','id','label_y','label_x'])
X_fusion_test=df_fusion_test.drop(columns=['image_id','id','label_y','label_x'])

y_train=df_fusion_train['label_y']
y_test=df_fusion_test['label_y']

X_total = np.concatenate((X_fusion_train, X_fusion_test), axis=0)
y_total = np.concatenate((y_train, y_test), axis=0)

In [ ]:
# Check for Label Consistency
labels_match_train = (df_fusion_train['label_x'] == df_fusion_train['label_y']).all()
labels_match_test = (df_fusion_test['label_x'] == df_fusion_test['label_y']).all()

if not labels_match_train or not labels_match_test:
    print("CRITICAL WARNING: Label Mismatch Detected!")
else:
    print("Labels are consistent.")

# FEATURE IMPORTANCE USING RANDOM FOREST CLASSIFIER

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


X_train_final = X_fusion_train
X_test_final = X_fusion_test

y_train_final = y_train
y_test_final = y_test


print("Training Random Forest on Fusion Features...")
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_final, y_train_final)


y_pred = rf.predict(X_test_final)
print("Classification Report:")
print(classification_report(y_test_final, y_pred))

#Feature Importance 
cols_to_drop = ['image_id', 'id', 'label_y', 'label_x', 'label']
feature_names = [col for col in df_fusion_train.columns if col not in cols_to_drop]

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

top_n = 15
top_indices = indices[:top_n]
top_features = [feature_names[i] for i in top_indices]
top_importances = importances[top_indices]

plt.figure(figsize=(10, 8))
plt.barh(range(top_n), top_importances, align='center')
plt.yticks(range(top_n), top_features)
plt.title("Top 15 Feature Importances (Fusion Model)")
plt.xlabel("Importance Score")
plt.gca().invert_yaxis() # Highest importance at the top
plt.tight_layout()
plt.show()

# SHAP AND LIME EXPLAINER

In [ ]:
import shap
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print("SHAP Plot")

# Fusion Classifier (Logistic Regression)
rf = LogisticRegression(max_iter=10000, solver='lbfgs', C=1.0, random_state=42, n_jobs=-1)
rf.fit(X_train_final, y_train_final)

explainer = shap.LinearExplainer(rf, X_train_final, feature_dependence="independent")
shap_values = explainer.shap_values(X_test_final)

# SHAP Summary Plots
shap.summary_plot(shap_values, X_test_final, plot_type="bar")   # Global importance
shap.summary_plot(shap_values, X_test_final)

In [ ]:
import pandas as pd
import lime
from lime import lime_tabular

def predict_fn(X_numpy):
    df = pd.DataFrame(X_numpy, columns=X_train_final.columns)
    return rf.predict_proba(df)

explainer = lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train_final), 
    feature_names=all_feature_names,  # Ensure this list matches column order!
    class_names=['Glioma', 'Meningioma', 'Notumor', 'Pituitary'],
    mode='classification'
)

idx = 0 
sample = X_test_final.iloc[idx]
true_label = y_test_final.iloc[idx] if hasattr(y_test_final, 'iloc') else y_test_final[idx]

print(f"Explaining Test Sample #{idx}")
print(f"True Label: {true_label}") 

exp = explainer.explain_instance(
    data_row=sample.values, 
    predict_fn=predict_fn,
    num_features=10, 
    top_labels=1
)


exp.show_in_notebook(show_table=True, show_all=False)

# GRAD-CAM VISUALISATION

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define Transforms
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load Data
train_dir = '/kaggle/input/brain-tumor-mri-dataset/Training'
train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

model = models.resnet50(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 4) # 4 Classes: Glioma, Meningioma, No Tumor, Pituitary
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print("Starting quick training (7 epochs)...")
for epoch in range(7):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/7 finished. Loss: {running_loss/len(train_loader):.4f}")

print("Training Complete. Model is ready for Grad-CAM.")


In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        target_layer.register_forward_hook(self.save_activation)

        target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, inp, out):
        self.activations = out

    def save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def __call__(self, x, class_idx=None):
        output = self.model(x)
        if class_idx is None:
            class_idx = output.argmax().item()

        self.model.zero_grad()
        output[:, class_idx].backward()

        gradients = self.gradients.detach().cpu().numpy()[0]
        activations = self.activations.detach().cpu().numpy()[0]

        weights = np.mean(gradients, axis=(1, 2))
        cam = np.sum(weights[:, np.newaxis, np.newaxis] * activations, axis=0)

        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (224, 224))
        cam = cam / cam.max()
        return cam

In [ ]:
model.eval()

#FOur times for each class
img_path = "/kaggle/input/brain-tumor-mri-dataset/Testing/pituitary/Te-pi_1.jpg"
img_pil = Image.open(img_path).convert('RGB')

img_tensor = data_transforms(img_pil).unsqueeze(0).to(device)

# Initialize Grad-CAM on the last convolutional layer of ResNet50
gradcam = GradCAM(model, model.layer4[2].conv3)

# Generate Heatmap
heatmap = gradcam(img_tensor)
output = model(img_tensor)
pred_idx = output.argmax().item()
print(f"Model Predicted Class Index: {pred_idx}") 

#just showing grad-cam overlay result
img_np = np.array(img_pil.resize((224, 224)))
heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
overlay = cv2.addWeighted(img_np, 0.6, heatmap_colored, 0.4, 0)
print("Pituitary")
plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title(f"Grad-CAM Result (Model Trained for 3 Epochs)")
plt.show()

In [ ]:
model.eval()

#FOur times for each class
img_path = "/kaggle/input/brain-tumor-mri-dataset/Testing/notumor/Te-no_1.jpg"
img_pil = Image.open(img_path).convert('RGB')

img_tensor = data_transforms(img_pil).unsqueeze(0).to(device)

# Initialize Grad-CAM on the last convolutional layer of ResNet50
gradcam = GradCAM(model, model.layer4[2].conv3)

# Generate Heatmap
heatmap = gradcam(img_tensor)
output = model(img_tensor)
pred_idx = output.argmax().item()
print(f"Model Predicted Class Index: {pred_idx}") 

#just showing grad-cam overlay result
img_np = np.array(img_pil.resize((224, 224)))
heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
overlay = cv2.addWeighted(img_np, 0.6, heatmap_colored, 0.4, 0)
print("No Tumor")
plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title(f"Grad-CAM Result (Model Trained for 3 Epochs)")
plt.show()

In [ ]:
model.eval()

#FOur times for each class
img_path = "/kaggle/input/brain-tumor-mri-dataset/Testing/meningioma/Te-aug-me_1.jpg"
img_pil = Image.open(img_path).convert('RGB')

img_tensor = data_transforms(img_pil).unsqueeze(0).to(device)

# Initialize Grad-CAM on the last convolutional layer of ResNet50
gradcam = GradCAM(model, model.layer4[2].conv3)

# Generate Heatmap
heatmap = gradcam(img_tensor)
output = model(img_tensor)
pred_idx = output.argmax().item()
print(f"Model Predicted Class Index: {pred_idx}") 

#just showing grad-cam overlay result
img_np = np.array(img_pil.resize((224, 224)))
heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
overlay = cv2.addWeighted(img_np, 0.6, heatmap_colored, 0.4, 0)
print("Meningioma")
plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title(f"Grad-CAM Result (Model Trained for 3 Epochs)")
plt.show()

In [ ]:
model.eval()

#FOur times for each class
img_path = "/kaggle/input/brain-tumor-mri-dataset/Testing/glioma/Te-gl_1.jpg"
img_pil = Image.open(img_path).convert('RGB')

img_tensor = data_transforms(img_pil).unsqueeze(0).to(device)

# Initialize Grad-CAM on the last convolutional layer of ResNet50
gradcam = GradCAM(model, model.layer4[2].conv3)

# Generate Heatmap
heatmap = gradcam(img_tensor)
output = model(img_tensor)
pred_idx = output.argmax().item()
print(f"Model Predicted Class Index: {pred_idx}") 

#just showing grad-cam overlay result
img_np = np.array(img_pil.resize((224, 224)))
heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
overlay = cv2.addWeighted(img_np, 0.6, heatmap_colored, 0.4, 0)
print("Glioma")
plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title(f"Grad-CAM Result (Model Trained for 3 Epochs)")
plt.show()